# ГП4 — fine-tune YOLO-pose на Waymo

GPU T4, Internet **On**, секрет `WANDB_API_KEY`, тот же zip картинок что в ГП2.

Код **не заливай файлами**. Ячейка ниже делает `git clone` / `git pull` с GitHub (ветка `missing-ml-2026`). После правок: `git push` → на Kaggle снова Run All.

1. Export всего сегмента в YOLO labels.
2. Fine-tune `yolov8s-pose`.
3. Eval `best.pt`.


In [ ]:
%pip install -q ultralytics pyarrow opencv-python-headless wandb hydra-core omegaconf pyyaml


Settings → Internet **On**. Репозиторий публичный, токен не нужен. Картинки — Add data (zip ГП2), код — только git.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/abbos-trnv/pose_estimation.git"
BRANCH = "missing-ml-2026"
REPO = Path("/kaggle/working/pose_estimation")

def _git(*args):
    subprocess.check_call(["git", *args])

if (REPO / ".git").exists():
    _git("-C", str(REPO), "fetch", "origin", BRANCH)
    _git("-C", str(REPO), "checkout", BRANCH)
    _git("-C", str(REPO), "pull", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO))

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("REPO", REPO, "HEAD")
subprocess.check_call(["git", "log", "-1", "--oneline"])

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print("wandb secret", e)


In [ ]:
!python scripts/export_yolo_pose.py data.max_frames=null


In [ ]:
!python scripts/train_pose.py train.epochs=20 train.imgsz=640 train.batch=8 train.workers=2 train.amp=true


In [ ]:
from pathlib import Path
best = sorted(Path("runs/train").glob("**/weights/best.pt"))
print(best)
w = best[-1] if best else "yolov8s-pose.pt"
print("eval weights", w)


In [ ]:
import subprocess
cmd = ["python", "scripts/eval_pose.py", f"model.weights={w}", "data.max_frames=null", "protocol=both"]
print(cmd)
subprocess.check_call(cmd)
